# 从 log 直接读取台站半径并可视化

这个 notebook **只读取 log 文件**，不再读取 ZHdata 目录。  
要求你的 log 中包含类似下面的行：

```text
RADIUS i=     1 lat=    21.1874 lon=   101.6963 radius=    0.315000 nper=    14
```

会输出：

1. 台站散点图（颜色表示半径）
2. 台站位置 + 半径圆
3. 半径直方图
4. 解析后的表格

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

In [ ]:
# ====== 改这里：你的 log 文件路径 ======
log_file = "/public/home/xiangyy/Project_ZHv2.0/real_run.log"

In [ ]:
def parse_radius_log(log_file):
    pattern = re.compile(
        r"RADIUS\s+i=\s*(?P<idx>\d+)\s+"
        r"lat=\s*(?P<lat>[-+]?\d*\.?\d+)\s+"
        r"lon=\s*(?P<lon>[-+]?\d*\.?\d+)\s+"
        r"radius=\s*(?P<radius>[-+]?\d*\.?\d+)\s+"
        r"nper=\s*(?P<nper>\d+)"
    )

    rows = []
    with open(log_file, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            m = pattern.search(line)
            if m:
                rows.append({
                    "idx": int(m.group("idx")),
                    "lat": float(m.group("lat")),
                    "lon": float(m.group("lon")),
                    "radius": float(m.group("radius")),
                    "nper": int(m.group("nper")),
                })

    if not rows:
        raise ValueError("没有在 log 中找到 RADIUS 行，请检查 log 格式。")

    df = pd.DataFrame(rows).sort_values("idx").reset_index(drop=True)
    return df

df = parse_radius_log(log_file)
print(f"读取到 {len(df)} 个台站")
df.head()

In [ ]:
# 半径统计
print(df[["radius"]].describe())

In [ ]:
# 图1：台站散点，颜色表示半径
plt.figure(figsize=(8, 6))
sc = plt.scatter(df["lon"], df["lat"], c=df["radius"], s=60)
plt.colorbar(sc, label="Radius")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Station locations colored by adaptive radius")
plt.axis("equal")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# 图2：台站位置 + 半径圆
fig, ax = plt.subplots(figsize=(9, 7))

ax.scatter(df["lon"], df["lat"], s=25)

for _, row in df.iterrows():
    circle = Circle((row["lon"], row["lat"]), row["radius"], fill=False, alpha=0.5)
    ax.add_patch(circle)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Stations and adaptive radii")
ax.axis("equal")
ax.grid(alpha=0.3)

# 自动加一点边界，避免圆被裁掉
xmin = (df["lon"] - df["radius"]).min()
xmax = (df["lon"] + df["radius"]).max()
ymin = (df["lat"] - df["radius"]).min()
ymax = (df["lat"] + df["radius"]).max()
dx = xmax - xmin
dy = ymax - ymin
ax.set_xlim(xmin - 0.05 * dx, xmax + 0.05 * dx)
ax.set_ylim(ymin - 0.05 * dy, ymax + 0.05 * dy)

plt.show()

In [ ]:
# 图3：半径分布
plt.figure(figsize=(7, 5))
plt.hist(df["radius"], bins=20)
plt.xlabel("Radius")
plt.ylabel("Count")
plt.title("Histogram of adaptive radii")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# 如果你想保存解析结果
# out_csv = Path(log_file).with_suffix(".radius.csv")
# df.to_csv(out_csv, index=False)
# print("已保存：", out_csv)
df